# Агент «Куда сходить в Алматы»

**Пайплайн:** sxodim.com → Jina Reader → GPT-5-mini (структурирование) → LlamaIndex RAG → агент

Логика живёт в `src/`, ноутбук её импортирует — так код не дублируется между `.py` и `.ipynb`.


## 0. Настройка

**В Colab** достаточно запустить ячейку ниже — она клонирует репозиторий (ноутбук импортирует из `src/`, поэтому одного `.ipynb` недостаточно), поставит зависимости и возьмёт ключ из панели Secrets (🔑 слева). Добавь туда `OPENAI_API_KEY` до запуска. GPU не нужен — хватит CPU runtime.

**Локально** ячейка ничего не делает: нужен `pip install -r requirements.txt` и `OPENAI_API_KEY` в `.env`.


In [1]:
REPO_URL = "https://github.com/zhadyrazhan/shodim-almaty-agent.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    import subprocess
    from pathlib import Path

    if not Path("shodim-almaty-agent").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    if Path("shodim-almaty-agent").exists():
        os.chdir("shodim-almaty-agent")

    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)

    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Colab: репозиторий и зависимости готовы, ключ загружен")
else:
    print("Локальный запуск — ключ берётся из .env")

Colab: репозиторий и зависимости готовы, ключ загружен


In [2]:
import json

from src import scraper, extract, agent
from src.config import SXODIM_DATA, RAW_DIR, LLM_BACKEND

print("LLM backend:", LLM_BACKEND)

LLM backend: openai


## 1. Парсинг сайта (Jina Reader)

Jina Reader отдаёт готовый markdown по любому URL, поэтому не нужно писать CSS-селекторы под вёрстку, которая может измениться.

In [3]:
written = scraper.scrape_all()
for name, path in written.items():
    print(f"{name}: {path.stat().st_size:,} байт")

fetching afisha: https://sxodim.com/almaty/afisha
  -> afisha.md (66,676 chars)
fetching weekend: https://sxodim.com/almaty/events/weekend
  -> weekend.md (67,979 chars)
fetching places: https://sxodim.com/almaty/places
  -> places.md (60,856 chars)
fetching main: https://sxodim.com/almaty
  [warn] main failed: HTTP Error 503: Service Unavailable
afisha: 93,048 байт
weekend: 94,908 байт
places: 85,909 байт


### Фрагмент спарсенных данных

In [4]:
raw = (RAW_DIR / "afisha.md").read_text(encoding="utf-8")
print(raw[:800])

Title: Мероприятия Алматы

URL Source: https://sxodim.com/almaty/afisha

Markdown Content:
[](https://sxodim.com/almaty)

Алматы

Русский Рус

[Қазақша](https://sxodim.com/locale/kk)[Русский](https://sxodim.com/locale/ru)[English](https://sxodim.com/locale/en)

[Билеты](https://sxodim.com/almaty/tickets)[Афиша](https://sxodim.com/almaty/afisha)[Журнал](https://sxodim.com/almaty/journal)[Места](https://sxodim.com/almaty/places)

[![Image 1](https://avatars.mds.yandex.net/get-adfox-content/2462621/260624_adfox_1070705_16070102.6e82ab4e25f7e64070a01903f30fecb3.png/optimize.webp)](https://yandex.ru/adfox/312864/clickURL?ad-session-id=4060611790192033410&adfox-version=1&duid=1790192032980723448&efc=1&esc=0&hash=0b139028f8343681&initial-engine-id=gnk&laas_region_id=ouq&p1=cfgjm&p2=glgo&p5=bjeiiw


## 2. Структурирование в JSON

Спарсенный markdown шумный (меню, реклама, дубли ссылок), поэтому он режется на чанки и передаётся модели со схемой Pydantic — structured output сам приводит всё к типам.

In [5]:
records = extract.extract_all()
print(f"\nвсего записей: {len(records)}")

3 pages -> 41 chunks
  [1/41] afisha.md: +8 (kept 8 unique)
  [2/41] afisha.md: +8 (kept 16 unique)
  [3/41] afisha.md: +3 (kept 19 unique)
  [4/41] afisha.md: +0 (kept 19 unique)
  [5/41] afisha.md: +0 (kept 19 unique)
  [6/41] afisha.md: +0 (kept 19 unique)
  [7/41] afisha.md: +0 (kept 19 unique)
  [8/41] afisha.md: +0 (kept 19 unique)
  [9/41] afisha.md: +0 (kept 19 unique)
  [10/41] afisha.md: +0 (kept 19 unique)
  [11/41] afisha.md: +0 (kept 19 unique)
  [12/41] afisha.md: +0 (kept 19 unique)
  [13/41] afisha.md: +29 (kept 48 unique)
  [14/41] afisha.md: +21 (kept 69 unique)
  [15/41] places.md: +10 (kept 79 unique)
  [16/41] places.md: +11 (kept 88 unique)
  [17/41] places.md: +0 (kept 88 unique)
  [18/41] places.md: +0 (kept 88 unique)
  [19/41] places.md: +0 (kept 88 unique)
  [20/41] places.md: +0 (kept 88 unique)
  [21/41] places.md: +0 (kept 88 unique)
  [22/41] places.md: +0 (kept 88 unique)
  [23/41] places.md: +0 (kept 88 unique)
  [24/41] places.md: +0 (kept 88 unique)
 

### Структурированный JSON (пример)

In [6]:
data = json.loads(SXODIM_DATA.read_text(encoding="utf-8"))
print(f"записей: {len(data)}\n")
print(json.dumps(data[:3], ensure_ascii=False, indent=2))

записей: 105

[
  {
    "title": "Большой летний фестиваль 2026",
    "kind": "event",
    "description": "4 июля в 17:00, Место проведения будет объявлено позже",
    "category": "Фестивали",
    "url": "https://sxodim.com/almaty/event/bolshoy-letniy-festival-2026",
    "date": "4 июля в 17:00",
    "price": "",
    "address": "",
    "good_for": [
      "компания друзей",
      "семья"
    ]
  },
  {
    "title": "Batyr Amanaty",
    "kind": "event",
    "description": "30 апреля в 19:00, Алматы Арена, мкр. Нуркент, 7",
    "category": "Концерты",
    "url": "https://sxodim.com/almaty/event/batyr-amanaty-zh-ne-kameraly-orkestr-lken-tribyut-koncert",
    "date": "30 апреля в 19:00",
    "price": "",
    "address": "Алматы Арена, мкр. Нуркент, 7",
    "good_for": [
      "компания друзей",
      "семья"
    ]
  },
  {
    "title": "Музыкальный фестиваль NonStop Music Fest",
    "kind": "event",
    "description": "3 мая в 19:00, Almaty Arena, мкр. Нуркент, 7",
    "category": "Фестивал

In [7]:
from collections import Counter
print("по типам:", dict(Counter(d["kind"] for d in data)))
print("по категориям:", dict(Counter(d["category"] for d in data).most_common(10)))

по типам: {'event': 41, 'place': 60, 'entertainment': 3, 'restaurant': 1}
по категориям: {'театр': 24, 'тур': 10, 'концерт': 9, 'туроператор/турагентство': 8, 'туризм': 7, 'Концерты': 5, 'развлечения/стендап': 5, 'Фестивали': 2, '': 2, 'развлечения/клуб': 2}


## 3. Агент (LlamaIndex + OpenAI)

Записи индексируются в `VectorStoreIndex`, поверх — query engine с системным промптом гида.

In [8]:
TEST_QUESTIONS = [
    "Куда сходить на выходных?",
    "Посоветуй место для свидания",
    "Куда сводить ребенка?",
    "Какие концерты будут?",
    "Где вкусно поесть?",
]

answers = {}
for q in TEST_QUESTIONS:
    a = agent.ask(q)
    answers[q] = a
    print(f"Q: {q}\nA: {a}\n{'-' * 70}")

Q: Куда сходить на выходных?
A: Отлично — вот что из афиши подходит для выходных:

- Тур «Озеро Иссык и водопад Медвежий» — внедорожный выезд на природу, старт от вашего отеля/аэропорта; отличный активный вариант для компании друзей или семьи. (Ко времени афиши: 13 сентября–31 декабря, цена 100 000 ₸.)
- Домики в горах «МегаДача» — загородные домики в горах, подходящее место для спокойного уик-энда с ночёвкой; адрес: с. Котырбулак, Талгарский район. (Цены в диапазоне 3 000–240 000 ₸.)
- Заявка на индивидуальный тур / туроператоры (ALTRAVEL, Enjoyers Travel) — если хочешь маршрут под себя или организовать поездку для одного, семьи или компании, можно оформить индивидуальный тур через заявку или обратиться к туроператору.

Скажи, какой формат ближе — активный день, спокойный загородный уик-энд или индивидуальный маршрут — помогу уточнить и выбрать.
----------------------------------------------------------------------
Q: Посоветуй место для свидания
A: Отлично — в афише есть два места, я

## 4. Сохранение примеров диалогов

Обязательный дилеверабл `agent_examples.md` — 5+ примеров.

In [9]:
lines = ["# Примеры диалогов с агентом\n"]
for q, a in answers.items():
    lines.append(f"\n## {q}\n\n{a}\n")

(agent.SXODIM_DATA.parent.parent / "agent_examples.md").write_text(
    "".join(lines), encoding="utf-8"
)
print(f"сохранено {len(answers)} диалогов в agent_examples.md")

сохранено 5 диалогов в agent_examples.md


## 5. Бонус: ORPO — делаем ответы дружелюбнее

Базовый агент отвечает корректно, но суховато. ORPO (Odds Ratio Preference Optimization) объединяет SFT и выравнивание по предпочтениям в один проход: не нужны ни отдельная reward-модель, ни reference-модель в памяти, поэтому всё помещается на бесплатный T4.

**Нужен GPU.** Runtime → Change runtime type → **T4 GPU**, затем Runtime → **Restart session** (смена типа без перезапуска не переносит сессию на GPU). Секции 1-4 выше работают и на CPU: если GPU нет, ячейки ниже сами себя пропустят.

In [16]:
import subprocess
import torch

HAS_GPU = torch.cuda.is_available()
print("CUDA available:", HAS_GPU)

if HAS_GPU:
    print("GPU:", torch.cuda.get_device_name(0))
    # Ставим только здесь, а не в секции 0: это ~2 ГБ пакетов, которые нужны
    # исключительно для ORPO. На CPU-прогоне секций 1-4 они только мешают.
    print("ставим зависимости для обучения...")
    subprocess.run(
        "pip install -q unsloth unsloth_zoo trl peft accelerate bitsandbytes datasets".split(),
        check=True,
    )
    print("готово")
else:
    print("GPU нет — секция 5 будет пропущена")

CUDA available: True
GPU: Tesla T4
ставим зависимости для обучения...
готово


### 5.1 Датасет предпочтений

ORPO нужны тройки (prompt, chosen, rejected). Обе стороны генерируются на **одних и тех же** записях афиши: `chosen` — тёплый ответ живым языком, `rejected` — сухая справка списком. Факты одинаковые, отличается только тон, значит модель учится именно стилю, а не содержанию.

Шаг идёт через OpenAI API и GPU не требует.

In [15]:
if HAS_GPU:
    !python training/build_preference_data.py --n 120

  [1/120] Куда сходить на выходных?
  [2/120] Посоветуй место для свидания
  [3/120] Куда сводить ребенка?
  [4/120] Какие концерты будут?
  [5/120] Где вкусно поесть?
  [6/120] Что нового открылось в Алматы?
  [7/120] Куда пойти с друзьями вечером?
  [8/120] Чем заняться в дождливый день?
  [9/120] Куда сходить одному?
  [10/120] Где провести время с семьей?
  [11/120] Есть что-нибудь бесплатное?
  [12/120] Посоветуй что-то необычное
  [13/120] Куда пойти после работы?
  [14/120] Где послушать живую музыку?
  [15/120] Куда сходить с родителями?
  [16/120] Куда сходить на выходных?
  [17/120] Посоветуй место для свидания
  [18/120] Куда сводить ребенка?
  [19/120] Какие концерты будут?
  [20/120] Где вкусно поесть?
  [21/120] Что нового открылось в Алматы?
  [22/120] Куда пойти с друзьями вечером?
  [23/120] Чем заняться в дождливый день?
  [24/120] Куда сходить одному?
  [25/120] Где провести время с семьей?
  [26/120] Есть что-нибудь бесплатное?
  [27/120] Посоветуй что-то необычное


In [17]:
import json
from pathlib import Path

TEST_QUESTIONS = [
    "Куда сходить на выходных?",
    "Посоветуй место для свидания",
    "Куда сводить ребенка?",
]

In [18]:
from pathlib import Path

if HAS_GPU:
    pairs = json.loads(Path("data/preference_data.json").read_text(encoding="utf-8"))
    print(f"пар: {len(pairs)}")
    p = pairs[0]
    print("\nВОПРОС:", p["prompt"])
    print("\n--- CHOSEN (тёплый) ---")
    print(p["chosen"][:500])
    print("\n--- REJECTED (сухой) ---")
    print(p["rejected"][:500])

пар: 120

ВОПРОС: Куда сходить на выходных?

--- CHOSEN (тёплый) ---
Круто, выходные — самое то для открытия чего-то нового в городе. Вот пару хороших идей из афиши Алматы, которые подойдут в разных настроениях:

1) Концерт CUPSIZE — Motor Club Almaty, 1 ноября, 20:00 (от 18 000 тг)
   - Кому подойдёт: компания друзей.
   - Почему стоит: живой концерт — отличный способ выпустить пар и провести вечер с энергетикой. Я люблю концерты именно за эту общую атмосферу — когда вся толпа подпевает и заряд бодрости на неделю вперёд.
   - Совет: берите удобную обувь и приход

--- REJECTED (сухой) ---
- Art Society — лекции
- Концерт группы CUPSIZE — концерт, от 18 000 тенге, 1 ноября в 20:00, Motor Club Almaty
- ТЮЗ им. Мусрепова — театр
- Театр «Керемет» — театр
- Волшебный киноопыт: Гарри Поттер и Философский Камень от vkuskino в Алматы — развлечение, киноужин с показом фильма «Гарри Поттер и Философский Камень»


### 5.2 Ответы ДО обучения

In [19]:
if HAS_GPU:
    from unsloth import FastLanguageModel

    BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
    model, tok = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=2048, load_in_4bit=True
    )

    def gen(m, t, question, max_new_tokens=220):
        prompt = t.apply_chat_template(
            [{"role": "user", "content": question}],
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = t([prompt], return_tensors="pt").to("cuda")
        out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        return t.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    FastLanguageModel.for_inference(model)
    before = {q: gen(model, tok, q) for q in TEST_QUESTIONS[:3]}
    for q, a in before.items():
        print(f"Q: {q}\nA: {a}\n{'-' * 70}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.11: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Куда сходить на выходных?
A: На выходные вы можете выбрать множество интересных мест для посещения в зависимости от ваших предпочтений и расположения. Вот несколько вариантов:

1. Парк или лес: Если у вас есть возможность побыть на свежем воздухе, то это отличное место для прогулки, катания на велосипеде или просто отдыха.

2. Музей: Посетить музей - отличный способ узнать больше о культуре и истории вашего города или региона.

3. Кинотеатр: Если вам нравятся фильмы, вы можете посетить кинотеатр и посмотреть новый фильм.

4. Театр: Посетите театр и посмотрите спектакль, который может быть интересен вашему вкусу.

5. Спа-салон: Если вы хотите расслабиться и восстановить силы, то сауну или бассейн могут быть отличным выбором.

6. Галерея или выставка
----------------------------------------------------------------------
Q: Посоветуй место для свидания
A: Выбор места для свидания зависит от ваших предпочтений и интересов. Вот несколько вариантов:

1. Парк: Если вы любите природу и спок

In [25]:
!git pull -q origin main

### 5.3 Обучение

Следим не только за падением loss, но и за **`rewards/margins`**: именно рост маржи показывает, что модель разводит тёплый и сухой ответы, а не просто подгоняется под оба.

In [26]:
if HAS_GPU:
    !python training/train_orpo.py --pairs data/preference_data.json --epochs 3

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
/content/shodim-almaty-agent/training/train_orpo.py:51: UserWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  from trl.experimental.orpo import ORPOConfig, ORPOTrainer
loaded 120 preference pairs
==((====))==  Unsloth 2026.9.11: Fast Qwen2 patching. Transformers: 5.5.0.
   \\  

### 5.4 Ответы ПОСЛЕ обучения

In [27]:
if HAS_GPU:
    from peft import PeftModel

    tuned, tuned_tok = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=2048, load_in_4bit=True
    )
    tuned = PeftModel.from_pretrained(tuned, "outputs/orpo-almaty")
    FastLanguageModel.for_inference(tuned)

    after = {q: gen(tuned, tuned_tok, q) for q in TEST_QUESTIONS[:3]}
    for q, a in after.items():
        print(f"Q: {q}\nA: {a}\n{'-' * 70}")

==((====))==  Unsloth 2026.9.11: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Куда сходить на выходных?
A: На выходные вы можете выбрать множество интересных мест для посещения в зависимости от ваших предпочтений и расположения. Вот несколько вариантов:

1. Парк или лес: Если вы любите природу, можно отправиться в ближайший парк или лес. Это отличный способ расслабиться и провести время с семьей или друзьями.

2. Музей: Посетить музей - это отличный способ узнать больше о культуре и истории вашего города или региона. Многие музеи предлагают бесплатные входные билеты или дешевые абонементы.

3. Театр или кинотеатр: Если вас интересуют фильмы или театральные представления, возможно, стоит посетить местный театр или кинофестиваль.

4. Спа-салон или сауна: Если вам нужен отдых и расслабление, можно просто пойти в сауну или спа-салон. 

5
----------------------------------------------------------------------
Q: Посоветуй место для свидания
A: Выбор места для свидания зависит от ваших предпочтений и интересов. Вот несколько вариантов:

1. Парк: Если вы любите приро

### 5.5 Сравнение ДО / ПОСЛЕ

Главный артефакт бонусной части: видно ли, что тон стал теплее.

In [28]:
if HAS_GPU:
    import pandas as pd

    df = pd.DataFrame(
        [{"вопрос": q, "ДО": before[q][:180], "ПОСЛЕ": after[q][:180]} for q in before]
    )
    pd.set_option("display.max_colwidth", 180)
    display(df)

    lines = ["# ORPO: ответы до и после\n"]
    for q in before:
        lines.append(f"\n## {q}\n\n**До:**\n\n{before[q]}\n\n**После:**\n\n{after[q]}\n")
    Path("orpo_examples.md").write_text("".join(lines), encoding="utf-8")
    print("\nсохранено в orpo_examples.md")

,вопрос,ДО,ПОСЛЕ
0,Куда сходить на выходных?,На выходные вы можете выбрать множество интересных мест для посещения в зависимости от ваших предпочтений и расположения. Вот несколько вариантов:\n\n1. Парк или лес: Если у ва...,На выходные вы можете выбрать множество интересных мест для посещения в зависимости от ваших предпочтений и расположения. Вот несколько вариантов:\n\n1. Парк или лес: Если вы л...
1,Посоветуй место для свидания,"Выбор места для свидания зависит от ваших предпочтений и интересов. Вот несколько вариантов:\n\n1. Парк: Если вы любите природу и спокойствие, парк может быть отличным выбором....","Выбор места для свидания зависит от ваших предпочтений и интересов. Вот несколько вариантов:\n\n1. Парк: Если вы любите природу и спокойствие, парк может быть отличным выбором...."
2,Куда сводить ребенка?,Выбор места для ребенка зависит от его возраста и интересов. Вот несколько вариантов:\n\n1. Парк или детский центр: Это отличное место для детей любого возраста. Они могут игра...,Выбор места для ребенка зависит от его возраста и интересов. Вот несколько вариантов:\n\n1. Парк или детский центр: Это отличное место для детей любого возраста. Они могут игра...



сохранено в orpo_examples.md


## 6. Скачать результаты

Файлы лежат внутри runtime и исчезнут вместе с сессией, поэтому забери их сразу. Сам ноутбук скачивается отдельно: File → Download → Download .ipynb — **после** того, как все ячейки отработали, чтобы выводы сохранились.

In [29]:
from pathlib import Path

ARTIFACTS = ["agent_examples.md", "data/sxodim_data.json", "orpo_examples.md"]

if IN_COLAB:
    from google.colab import files

    for path in ARTIFACTS:
        # orpo_examples.md only exists if section 5 ran (needs a GPU).
        if Path(path).exists():
            files.download(path)
        else:
            print(f"{path}: пропущен (не создан)")
else:
    for path in ARTIFACTS:
        p = Path(path)
        print(f"{path}: {'есть' if p.exists() else 'НЕТ'}"
              f"{f' ({p.stat().st_size:,} байт)' if p.exists() else ''}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>